# Evaluate the SpaceParts agent team

Run this **in a Fabric notebook** in the workspace that holds the agents (the evaluation SDK is preview).
It asks each published agent every question in its `ground_truth_<agent>.csv`, three times, and an LLM judge compares each answer with the SQL-verified expected answer from `verify_answers.sql`.

Upload the three `ground_truth_*.csv` files to the lakehouse **Files/evaluation/** folder first (or paste them into the cell below).
Only questions that pass **every** repeat go on stage as chips.


In [ ]:
%pip install -q fabric-data-agent-sdk


In [ ]:
import pandas as pd
from fabric.dataagent.evaluation import evaluate_data_agent, get_evaluation_summary, get_evaluation_details

AGENTS = {
    'sales': 'SpaceParts Sales Analyst',
    'operations': 'SpaceParts Operations',
    'planning': 'SpaceParts Planning',
}
GROUND_TRUTH = '/lakehouse/default/Files/evaluation/ground_truth_{key}.csv'


The critic prompt is **lenient about formatting and strict about numbers**: the answer passes only if every figure in the expected answer appears (rounded to the nearest whole euro or 0.1%) and nothing contradicts it.


In [ ]:
critic_prompt = '''
Given the following query, expected answer, and actual answer, please determine if the actual answer is equivalent to expected answer.
Ignore formatting, ordering of columns, currency symbols, thousands separators and extra explanation.
Every number in the expected answer must appear in the actual answer, allowing rounding to the nearest whole euro or 0.1 percentage point.
Names (regions, key accounts, managers, product types, brands) must match.
If the actual answer uses a different period or currency than the expected answer, it is not equivalent.
Query: {query}
Expected Answer: {expected_answer}
Actual Answer: {actual_answer}
If the actual answer is equivalent to the expected answer, respond with 'Yes'. Otherwise respond with 'No'. If unclear, respond with 'Unclear'.
'''


In [ ]:
results = {}
for key, agent in AGENTS.items():
    df = pd.read_csv(GROUND_TRUTH.format(key=key))
    print(f'{agent}: {len(df)} questions')
    results[key] = evaluate_data_agent(
        df,
        agent,
        table_name=f'spaceparts_evaluation_{key}',
        num_query_repeats=3,          # ask each question 3 times; agents are non-deterministic
        critic_prompt=critic_prompt,
        data_agent_stage='production',  # the published version, the one callers get
    )


In [ ]:
for key in AGENTS:
    print(key)
    display(get_evaluation_summary(f'spaceparts_evaluation_{key}'))


## Read the failures
Open the details for any question that isn't `Yes` every time. The `steps` show the SQL the agent wrote; compare it with `verify_answers.sql` and fix **one layer** (tables, instructions, or an example query), then re-run everything.


In [ ]:
details = get_evaluation_details(results['sales'], 'spaceparts_evaluation_sales', get_all_rows=True, verbose=True)
